# Stage 6: Excel Export

Export all metrics to Excel with real-world unit conversions.

**Input**: All JSON files in `metrics/`  
**Output**: `output.xlsx`

In [ ]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
from utils import load_metrics, VOXEL_SIZE_MM, VOXEL_VOLUME_MM3

In [ ]:
# Setup directories
target = Path(TARGET_DIR)
METRICS_DIR = target / "metrics"
OUTPUT_FILE = target / "output.xlsx"

print(f"Metrics directory: {METRICS_DIR}")
print(f"Output file: {OUTPUT_FILE}")
print(f"\nVoxel size: {VOXEL_SIZE_MM} mm ({VOXEL_SIZE_MM * 1000:.0f} micrometers)")
print(f"Voxel volume: {VOXEL_VOLUME_MM3:.2e} mm^3")

In [ ]:
# Load all available metrics
metrics_files = {
    'hull': METRICS_DIR / "hull_metrics.json",
    'shrink': METRICS_DIR / "shrink_metrics.json",
    'axis': METRICS_DIR / "axis_info.json",
    'socket': METRICS_DIR / "socket_metrics.json",
    'bone_v1': METRICS_DIR / "bone_length_metrics_v1.json",
    'bone_v2': METRICS_DIR / "bone_length_metrics_v2.json",
}

metrics_data = {}
for name, path in metrics_files.items():
    if path.exists():
        metrics_data[name] = load_metrics(path)
        print(f"  Loaded {name}: {len(metrics_data[name])} records")
    else:
        print(f"  {name}: Not found")

In [ ]:
def create_combined_dataframe(metrics_data):
    """Create combined DataFrame from all metrics sources."""
    
    # Use socket metrics as base (has sample_name)
    if 'socket' not in metrics_data:
        print("ERROR: Socket metrics required")
        return None
    
    socket_data = metrics_data['socket']
    
    # Create lookup dicts
    bone_v1_dict = {}
    if 'bone_v1' in metrics_data:
        for item in metrics_data['bone_v1']:
            name = item.get('sample_name', '')
            bone_v1_dict[name] = item
            bone_v1_dict[name + '.nii'] = item
    
    bone_v2_dict = {}
    if 'bone_v2' in metrics_data:
        for item in metrics_data['bone_v2']:
            name = item.get('sample_name', '')
            bone_v2_dict[name] = item
            bone_v2_dict[name + '.nii'] = item
    
    # Build combined rows
    rows = []
    for socket_item in socket_data:
        sample_name = socket_item.get('sample_name', '')
        
        socket_vol = socket_item.get('socket_volume', 0)
        socket_rad = socket_item.get('equivalent_radius')
        phalanx_vol = socket_item.get('shrunk_volume', 0)
        
        row = {
            'Sample': sample_name,
            
            # Socket metrics
            'Socket_Volume_voxels': socket_vol,
            'Socket_Volume_mm3': socket_vol * VOXEL_VOLUME_MM3 if socket_vol else None,
            'Socket_Radius_voxels': socket_rad,
            'Socket_Radius_mm': socket_rad * VOXEL_SIZE_MM if socket_rad else None,
            
            # Phalanx volume
            'Phalanx_Volume_voxels': phalanx_vol,
            'Phalanx_Volume_mm3': phalanx_vol * VOXEL_VOLUME_MM3 if phalanx_vol else None,
        }
        
        # Add centroid
        centroid = socket_item.get('centroid')
        if centroid:
            row['Socket_COM_X_voxels'] = centroid[0]
            row['Socket_COM_Y_voxels'] = centroid[1]
            row['Socket_COM_Z_voxels'] = centroid[2]
        
        # Add bone length v1 metrics
        if sample_name in bone_v1_dict:
            b = bone_v1_dict[sample_name]
            bone_len = b.get('bone_length_euclidean', 0)
            row['Bone_Length_v1_voxels'] = bone_len
            row['Bone_Length_v1_mm'] = bone_len * VOXEL_SIZE_MM if bone_len else None
            row['Euclidean_Distance_v1_voxels'] = b.get('euclidean_distance')
            row['BV_TV_phalanx'] = b.get('BV_TV_phalanx')
            row['BV_TV_groundtruth'] = b.get('BV_TV_groundtruth')
        
        # Add bone length v2 metrics
        if sample_name in bone_v2_dict:
            b = bone_v2_dict[sample_name]
            bone_len = b.get('bone_length_euclidean', 0)
            socket_len = b.get('socket_length_euclidean', 0)
            row['Bone_Length_v2_voxels'] = bone_len
            row['Bone_Length_v2_mm'] = bone_len * VOXEL_SIZE_MM if bone_len else None
            row['Socket_Segment_v2_voxels'] = socket_len
            row['Socket_Segment_v2_mm'] = socket_len * VOXEL_SIZE_MM if socket_len else None
        
        rows.append(row)
    
    return pd.DataFrame(rows)


def create_summary_stats(df):
    """Create summary statistics DataFrame."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    stats = []
    for col in numeric_cols:
        data = df[col].dropna()
        if len(data) > 0:
            stats.append({
                'Metric': col,
                'Count': len(data),
                'Mean': data.mean(),
                'Std': data.std(),
                'Min': data.min(),
                'Q25': data.quantile(0.25),
                'Median': data.median(),
                'Q75': data.quantile(0.75),
                'Max': data.max(),
            })
    
    return pd.DataFrame(stats)


def create_length_volume_comparison(metrics_data):
    """Create simplified DataFrame with bone lengths and volumes only."""
    
    if 'socket' not in metrics_data:
        return None
    
    socket_data = metrics_data['socket']
    
    # Create lookup dicts
    bone_v1_dict = {}
    if 'bone_v1' in metrics_data:
        for item in metrics_data['bone_v1']:
            name = item.get('sample_name', '')
            bone_v1_dict[name] = item
    
    bone_v2_dict = {}
    if 'bone_v2' in metrics_data:
        for item in metrics_data['bone_v2']:
            name = item.get('sample_name', '')
            bone_v2_dict[name] = item
    
    rows = []
    for socket_item in socket_data:
        sample_name = socket_item.get('sample_name', '')
        
        row = {
            'Sample': sample_name,
        }
        
        # Bone length v1
        if sample_name in bone_v1_dict:
            b = bone_v1_dict[sample_name]
            row['Bone_Length_v1_voxels'] = b.get('bone_length_euclidean', 0)
            row['Bone_Length_v1_mm'] = b.get('bone_length_euclidean', 0) * VOXEL_SIZE_MM
        else:
            row['Bone_Length_v1_voxels'] = None
            row['Bone_Length_v1_mm'] = None
        
        # Bone length v2
        if sample_name in bone_v2_dict:
            b = bone_v2_dict[sample_name]
            row['Bone_Length_v2_voxels'] = b.get('bone_length_euclidean', 0)
            row['Bone_Length_v2_mm'] = b.get('bone_length_euclidean', 0) * VOXEL_SIZE_MM
            row['Socket_Volume_voxels'] = b.get('socket_volume', 0)
            row['Socket_Volume_mm3'] = b.get('socket_volume', 0) * VOXEL_VOLUME_MM3
            row['Bone_Volume_voxels'] = b.get('bone_volume', 0)
            row['Bone_Volume_mm3'] = b.get('bone_volume', 0) * VOXEL_VOLUME_MM3
        else:
            row['Bone_Length_v2_voxels'] = None
            row['Bone_Length_v2_mm'] = None
            # Fall back to socket metrics for volumes
            row['Socket_Volume_voxels'] = socket_item.get('socket_volume', 0)
            row['Socket_Volume_mm3'] = socket_item.get('socket_volume', 0) * VOXEL_VOLUME_MM3
            row['Bone_Volume_voxels'] = socket_item.get('shrunk_volume', 0)
            row['Bone_Volume_mm3'] = socket_item.get('shrunk_volume', 0) * VOXEL_VOLUME_MM3
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [ ]:
# Create combined DataFrame
print("\nCombining metrics...")
df = create_combined_dataframe(metrics_data)

if df is not None:
    print(f"Combined: {len(df)} samples, {len(df.columns)} columns")
    
    # Create summary stats
    print("Computing statistics...")
    stats_df = create_summary_stats(df)
    
    # Create length/volume comparison sheet
    print("Creating length/volume comparison...")
    comparison_df = create_length_volume_comparison(metrics_data)
    
    # Export to Excel
    print(f"\nExporting to {OUTPUT_FILE}...")
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='All_Measurements', index=False)
        stats_df.to_excel(writer, sheet_name='Summary_Statistics', index=False)
        if comparison_df is not None:
            comparison_df.to_excel(writer, sheet_name='Length_Volume', index=False)
        
        # Format worksheets
        for sheet_name in writer.sheets.keys():
            worksheet = writer.sheets[sheet_name]
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 40)
                worksheet.column_dimensions[column_letter].width = adjusted_width
    
    print(f"\n[OK] Excel file created: {OUTPUT_FILE}")

In [ ]:
# Summary
print("\n" + "="*70)
print("EXPORT COMPLETE")
print("="*70)

print(f"\nSheets created:")
print(f"  1. All_Measurements - Complete data for all samples")
print(f"  2. Summary_Statistics - Descriptive statistics")
print(f"  3. Length_Volume - Bone lengths (v1, v2) and volumes")

if df is not None and len(df) > 0:
    print(f"\nSamples: {len(df)}")
    
    # Print key metrics if available
    if 'Socket_Radius_voxels' in df.columns:
        print(f"\nSocket radius:")
        print(f"  Voxels: {df['Socket_Radius_voxels'].mean():.2f} +/- {df['Socket_Radius_voxels'].std():.2f}")
        print(f"  mm: {df['Socket_Radius_mm'].mean():.4f} +/- {df['Socket_Radius_mm'].std():.4f}")
    
    if 'Bone_Length_v1_voxels' in df.columns and df['Bone_Length_v1_voxels'].notna().any():
        print(f"\nBone length (v1):")
        print(f"  Voxels: {df['Bone_Length_v1_voxels'].mean():.2f} +/- {df['Bone_Length_v1_voxels'].std():.2f}")
        print(f"  mm: {df['Bone_Length_v1_mm'].mean():.4f} +/- {df['Bone_Length_v1_mm'].std():.4f}")
    
    if 'Bone_Length_v2_voxels' in df.columns and df['Bone_Length_v2_voxels'].notna().any():
        print(f"\nBone length (v2):")
        print(f"  Voxels: {df['Bone_Length_v2_voxels'].mean():.2f} +/- {df['Bone_Length_v2_voxels'].std():.2f}")
        print(f"  mm: {df['Bone_Length_v2_mm'].mean():.4f} +/- {df['Bone_Length_v2_mm'].std():.4f}")

print(f"\nOutput: {OUTPUT_FILE}")